# Reproducible 80/20 stratified random split

This notebook splits `0_data/outputs/combined_data.parquet` into training and test sets using:

- **80% training / 20% test**;
- fixed random seed **2026**;
- stratification by melting-point class;
- low MP: `MP < 250 °C`;
- high MP: `MP >= 250 °C`.

The split is performed at the row level after dataset curation. Because `combined_data.parquet` contains one row per unique canonical SMILES, the same compound cannot occur in both sets. No scaling, feature selection, resampling, augmentation, or class balancing is performed here.

Run `0_data/combine_data_process.ipynb` first whenever source data or curation rules change.

In [1]:
from pathlib import Path
import hashlib

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from IPython.display import display
from sklearn.model_selection import train_test_split

RANDOM_STATE = 2026
TEST_SIZE = 0.20
HIGH_MP_THRESHOLD_C = 250.0
SPLIT_METHOD = 'stratified_random_80_20'

project_root_candidates = [Path.cwd(), Path.cwd().parent]
PROJECT_ROOT = next((
    path.resolve() for path in project_root_candidates
    if (path / '0_data' / 'outputs' / 'combined_data.parquet').is_file()
), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        'Could not locate 0_data/outputs/combined_data.parquet. Run this notebook '
        'from the repository root or from 1_data_split.'
    )

INPUT_PATH = PROJECT_ROOT / '0_data' / 'outputs' / 'combined_data.parquet'
OUTPUT_DIR = (
    PROJECT_ROOT / '1_data_split' / 'outputs'
    / f'stratified_random_seed{RANDOM_STATE}'
)
TRAIN_PATH = OUTPUT_DIR / 'train.parquet'
TEST_PATH = OUTPUT_DIR / 'test.parquet'
MANIFEST_PATH = OUTPUT_DIR / 'split_manifest.csv'
SUMMARY_PATH = OUTPUT_DIR / 'split_summary.csv'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def sha256(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b''):
            digest.update(chunk)
    return digest.hexdigest()

print(f'Input: {INPUT_PATH}')
print(f'Output directory: {OUTPUT_DIR}')
print(f'Random seed: {RANDOM_STATE}')

Input: /Users/sdl5_mp/Documents/GitHub/melting_point_2026/0_data/outputs/combined_data.parquet
Output directory: /Users/sdl5_mp/Documents/GitHub/melting_point_2026/1_data_split/outputs/stratified_random_seed2026
Random seed: 2026


## Load and validate the curated dataset

The source-label check intentionally stops the workflow if the saved combined dataset is stale or incomplete.

In [2]:
combined_table = pq.read_table(INPUT_PATH)
combined_data = combined_table.to_pandas()
input_sha256 = sha256(INPUT_PATH)

required_columns = {'SMILES', 'MP', 'Source'}
missing_columns = sorted(required_columns.difference(combined_data.columns))
if missing_columns:
    raise ValueError(f'Combined dataset is missing required columns: {missing_columns}')

assert combined_data['SMILES'].notna().all()
assert combined_data['MP'].notna().all()
assert combined_data['SMILES'].is_unique
assert combined_data['MP'].between(0, 500, inclusive='both').all()

observed_sources = set()
for source_values in combined_data['Source']:
    observed_sources.update(map(int, source_values))
expected_sources = {1, 2, 3, 4}
if observed_sources != expected_sources:
    raise ValueError(
        f'Expected source labels {sorted(expected_sources)}, found '
        f'{sorted(observed_sources)}. Rerun 0_data/combine_data_process.ipynb '
        'before creating a permanent split.'
    )

combined_data['MP_class'] = np.where(
    combined_data['MP'].ge(HIGH_MP_THRESHOLD_C), 'high', 'low'
)
class_overview = (
    combined_data['MP_class'].value_counts().reindex(['low', 'high'], fill_value=0)
    .rename_axis('MP_class').reset_index(name='Compounds')
)
class_overview['Percent'] = 100 * class_overview['Compounds'] / len(combined_data)
display(class_overview)
print(f'Validated compounds: {len(combined_data):,}')
print(f'Input SHA256: {input_sha256}')

,MP_class,Compounds,Percent
0,low,281994,92.606434
1,high,22514,7.393566


Validated compounds: 304,508
Input SHA256: 6dc1a218d02f0a561674995471ebb38f1109322eb73b88bbaf24e1ba2abc6db0


## Create the fixed stratified split

Only the low/high MP label is used for stratification. Consensus method, quality tier, source support, and molecular properties remain available for post-split balance checks but do not control the assignment.

In [3]:
row_indices = np.arange(len(combined_data))
train_indices, test_indices = train_test_split(
    row_indices,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    shuffle=True,
    stratify=combined_data['MP_class'],
)

# Verify that repeating the operation with the same seed produces identical assignments.
repeat_train_indices, repeat_test_indices = train_test_split(
    row_indices,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    shuffle=True,
    stratify=combined_data['MP_class'],
)
assert np.array_equal(train_indices, repeat_train_indices)
assert np.array_equal(test_indices, repeat_test_indices)

train_data = combined_data.iloc[train_indices].copy()
test_data = combined_data.iloc[test_indices].copy()
train_data['Split'] = 'train'
test_data['Split'] = 'test'
train_data = train_data.sort_values('SMILES', kind='stable').reset_index(drop=True)
test_data = test_data.sort_values('SMILES', kind='stable').reset_index(drop=True)

assert set(train_data['SMILES']).isdisjoint(set(test_data['SMILES']))
assert len(train_data) + len(test_data) == len(combined_data)
assert set(train_data['SMILES']) | set(test_data['SMILES']) == set(combined_data['SMILES'])

print(f'Training compounds: {len(train_data):,} ({100 * len(train_data) / len(combined_data):.2f}%)')
print(f'Test compounds: {len(test_data):,} ({100 * len(test_data) / len(combined_data):.2f}%)')

Training compounds: 243,606 (80.00%)
Test compounds: 60,902 (20.00%)


## Write the split outputs

`train.parquet` and `test.parquet` retain every combined-data column and add `MP_class` and `Split`. The CSV manifest is the authoritative compound-to-split assignment.

In [4]:
def write_parquet_with_source_schema(frame, output_path):
    table = pa.Table.from_pandas(frame, preserve_index=False)
    source_index = table.column_names.index('Source')
    table = table.set_column(
        source_index,
        'Source',
        pa.array(frame['Source'].tolist(), type=pa.list_(pa.int8())),
    )
    temporary_path = output_path.with_name(f'.{output_path.name}.tmp')
    pq.write_table(table, temporary_path, compression='zstd')
    temporary_path.replace(output_path)

manifest = pd.concat([
    train_data[['SMILES', 'Split', 'MP_class']],
    test_data[['SMILES', 'Split', 'MP_class']],
], ignore_index=True).sort_values('SMILES', kind='stable').reset_index(drop=True)
manifest['Random_state'] = RANDOM_STATE
manifest['Split_method'] = SPLIT_METHOD

summary_rows = []
for split_name, frame in [('all', combined_data), ('train', train_data), ('test', test_data)]:
    counts = frame['MP_class'].value_counts().reindex(['low', 'high'], fill_value=0)
    for mp_class, count in counts.items():
        summary_rows.append({
            'Split': split_name,
            'MP_class': mp_class,
            'Compounds': int(count),
            'Percent_within_split': 100 * int(count) / len(frame),
            'Total_compounds_in_split': len(frame),
            'Train_fraction': 1 - TEST_SIZE,
            'Test_fraction': TEST_SIZE,
            'High_MP_threshold_C': HIGH_MP_THRESHOLD_C,
            'Random_state': RANDOM_STATE,
            'Split_method': SPLIT_METHOD,
            'Input_file': str(INPUT_PATH.relative_to(PROJECT_ROOT)),
            'Input_SHA256': input_sha256,
        })
split_summary = pd.DataFrame(summary_rows)

write_parquet_with_source_schema(train_data, TRAIN_PATH)
write_parquet_with_source_schema(test_data, TEST_PATH)

temporary_manifest = MANIFEST_PATH.with_name(f'.{MANIFEST_PATH.name}.tmp')
temporary_summary = SUMMARY_PATH.with_name(f'.{SUMMARY_PATH.name}.tmp')
manifest.to_csv(temporary_manifest, index=False)
split_summary.to_csv(temporary_summary, index=False)
temporary_manifest.replace(MANIFEST_PATH)
temporary_summary.replace(SUMMARY_PATH)

display(split_summary.round({'Percent_within_split': 4}))
print(f'Wrote: {TRAIN_PATH}')
print(f'Wrote: {TEST_PATH}')
print(f'Wrote: {MANIFEST_PATH}')
print(f'Wrote: {SUMMARY_PATH}')

,Split,MP_class,Compounds,Percent_within_split,Total_compounds_in_split,Train_fraction,Test_fraction,High_MP_threshold_C,Random_state,Split_method,Input_file,Input_SHA256
0,all,low,281994,92.6064,304508,0.8,0.2,250.0,2026,stratified_random_80_20,0_data/outputs/combined_data.parquet,6dc1a218d02f0a561674995471ebb38f1109322eb73b88...
1,all,high,22514,7.3936,304508,0.8,0.2,250.0,2026,stratified_random_80_20,0_data/outputs/combined_data.parquet,6dc1a218d02f0a561674995471ebb38f1109322eb73b88...
2,train,low,225595,92.6065,243606,0.8,0.2,250.0,2026,stratified_random_80_20,0_data/outputs/combined_data.parquet,6dc1a218d02f0a561674995471ebb38f1109322eb73b88...
3,train,high,18011,7.3935,243606,0.8,0.2,250.0,2026,stratified_random_80_20,0_data/outputs/combined_data.parquet,6dc1a218d02f0a561674995471ebb38f1109322eb73b88...
4,test,low,56399,92.6062,60902,0.8,0.2,250.0,2026,stratified_random_80_20,0_data/outputs/combined_data.parquet,6dc1a218d02f0a561674995471ebb38f1109322eb73b88...
5,test,high,4503,7.3938,60902,0.8,0.2,250.0,2026,stratified_random_80_20,0_data/outputs/combined_data.parquet,6dc1a218d02f0a561674995471ebb38f1109322eb73b88...


Wrote: /Users/sdl5_mp/Documents/GitHub/melting_point_2026/1_data_split/outputs/stratified_random_seed2026/train.parquet
Wrote: /Users/sdl5_mp/Documents/GitHub/melting_point_2026/1_data_split/outputs/stratified_random_seed2026/test.parquet
Wrote: /Users/sdl5_mp/Documents/GitHub/melting_point_2026/1_data_split/outputs/stratified_random_seed2026/split_manifest.csv
Wrote: /Users/sdl5_mp/Documents/GitHub/melting_point_2026/1_data_split/outputs/stratified_random_seed2026/split_summary.csv


## Validate the saved split

These checks confirm disjointness, completeness, class-ratio preservation, schema, metadata, and exact agreement between the manifest and Parquet files.

In [5]:
saved_train_table = pq.read_table(TRAIN_PATH)
saved_test_table = pq.read_table(TEST_PATH)
saved_train = saved_train_table.to_pandas()
saved_test = saved_test_table.to_pandas()
saved_manifest = pd.read_csv(MANIFEST_PATH)
saved_summary = pd.read_csv(SUMMARY_PATH)

assert saved_train['SMILES'].is_unique
assert saved_test['SMILES'].is_unique
assert set(saved_train['SMILES']).isdisjoint(set(saved_test['SMILES']))
assert len(saved_train) + len(saved_test) == len(combined_data)
assert set(saved_train['SMILES']) | set(saved_test['SMILES']) == set(combined_data['SMILES'])
assert saved_train['Split'].eq('train').all()
assert saved_test['Split'].eq('test').all()
assert saved_train['MP_class'].eq(np.where(saved_train['MP'].ge(HIGH_MP_THRESHOLD_C), 'high', 'low')).all()
assert saved_test['MP_class'].eq(np.where(saved_test['MP'].ge(HIGH_MP_THRESHOLD_C), 'high', 'low')).all()
assert pa.types.is_list(saved_train_table.schema.field('Source').type)
assert pa.types.is_int8(saved_train_table.schema.field('Source').type.value_type)
assert pa.types.is_list(saved_test_table.schema.field('Source').type)
assert pa.types.is_int8(saved_test_table.schema.field('Source').type.value_type)

parquet_assignments = pd.concat([
    saved_train[['SMILES', 'Split', 'MP_class']],
    saved_test[['SMILES', 'Split', 'MP_class']],
], ignore_index=True).sort_values('SMILES', kind='stable').reset_index(drop=True)
manifest_assignments = saved_manifest[
    ['SMILES', 'Split', 'MP_class']
].sort_values('SMILES', kind='stable').reset_index(drop=True)
assert parquet_assignments.equals(manifest_assignments)
assert saved_manifest['Random_state'].eq(RANDOM_STATE).all()
assert saved_manifest['Split_method'].eq(SPLIT_METHOD).all()
assert saved_summary['Random_state'].eq(RANDOM_STATE).all()
assert saved_summary['Input_SHA256'].eq(input_sha256).all()

overall_high_fraction = combined_data['MP_class'].eq('high').mean()
train_high_fraction = saved_train['MP_class'].eq('high').mean()
test_high_fraction = saved_test['MP_class'].eq('high').mean()
assert abs(train_high_fraction - overall_high_fraction) < 1e-4
assert abs(test_high_fraction - overall_high_fraction) < 1e-4

display(saved_summary.round({'Percent_within_split': 4}))
print(f'Manifest SHA256: {sha256(MANIFEST_PATH)}')
print('All saved-split validation checks passed.')

,Split,MP_class,Compounds,Percent_within_split,Total_compounds_in_split,Train_fraction,Test_fraction,High_MP_threshold_C,Random_state,Split_method,Input_file,Input_SHA256
0,all,low,281994,92.6064,304508,0.8,0.2,250.0,2026,stratified_random_80_20,0_data/outputs/combined_data.parquet,6dc1a218d02f0a561674995471ebb38f1109322eb73b88...
1,all,high,22514,7.3936,304508,0.8,0.2,250.0,2026,stratified_random_80_20,0_data/outputs/combined_data.parquet,6dc1a218d02f0a561674995471ebb38f1109322eb73b88...
2,train,low,225595,92.6065,243606,0.8,0.2,250.0,2026,stratified_random_80_20,0_data/outputs/combined_data.parquet,6dc1a218d02f0a561674995471ebb38f1109322eb73b88...
3,train,high,18011,7.3935,243606,0.8,0.2,250.0,2026,stratified_random_80_20,0_data/outputs/combined_data.parquet,6dc1a218d02f0a561674995471ebb38f1109322eb73b88...
4,test,low,56399,92.6062,60902,0.8,0.2,250.0,2026,stratified_random_80_20,0_data/outputs/combined_data.parquet,6dc1a218d02f0a561674995471ebb38f1109322eb73b88...
5,test,high,4503,7.3938,60902,0.8,0.2,250.0,2026,stratified_random_80_20,0_data/outputs/combined_data.parquet,6dc1a218d02f0a561674995471ebb38f1109322eb73b88...


Manifest SHA256: b5ce36f64c180163e2b7cbf6597fe98c2fb5871ab15ef27a434b3af86c5eed0f
All saved-split validation checks passed.
